In [1]:
import pandas as pd

In [2]:
# Criando nossos dataframes
emails = pd.read_excel(r'Bases de Dados\Emails.xlsx')
lojas = pd.read_csv(r'Bases de Dados\Lojas.csv', encoding='latin1', sep=';')
vendas = pd.read_excel(r'Bases de Dados\Vendas.xlsx')

display(emails.head())
display(lojas.head())
display(vendas.head())

,Loja,Gerente,E-mail
0,Iguatemi Esplanada,Helena,pythonimpressionador+helena@gmail.com
1,Shopping Midway Mall,Alice,pythonimpressionador+alice@gmail.com
2,Norte Shopping,Laura,pythonimpressionador+laura@gmail.com
3,Shopping Iguatemi Fortaleza,Manuela,pythonimpressionador+manuela@gmail.com
4,Shopping União de Osasco,Valentina,pythonimpressionador+valentina@gmail.com


,ID Loja,Loja
0,1,Iguatemi Esplanada
1,2,Shopping Midway Mall
2,3,Norte Shopping
3,4,Shopping Iguatemi Fortaleza
4,5,Shopping União de Osasco


,Código Venda,Data,ID Loja,Produto,Quantidade,Valor Unitário,Valor Final
0,1,2019-01-01,1,Sapato Estampa,1,358,358
1,1,2019-01-01,1,Camiseta,2,180,360
2,1,2019-01-01,1,Sapato Xadrez,1,368,368
3,2,2019-01-02,3,Relógio,3,200,600
4,2,2019-01-02,3,Chinelo Liso,1,71,71


In [3]:
#tratando a base de vendas e trazendo o nome loja
vendas = pd.merge(vendas, lojas[['ID Loja', 'Loja']], on='ID Loja', how='left')

display(vendas.head())

,Código Venda,Data,ID Loja,Produto,Quantidade,Valor Unitário,Valor Final,Loja
0,1,2019-01-01,1,Sapato Estampa,1,358,358,Iguatemi Esplanada
1,1,2019-01-01,1,Camiseta,2,180,360,Iguatemi Esplanada
2,1,2019-01-01,1,Sapato Xadrez,1,368,368,Iguatemi Esplanada
3,2,2019-01-02,3,Relógio,3,200,600,Norte Shopping
4,2,2019-01-02,3,Chinelo Liso,1,71,71,Norte Shopping


In [4]:
#Criando uma tabela para cada loja

dicionario_lojas = {}
for loja in lojas['Loja']:
    dicionario_lojas[loja] = vendas.loc[vendas['Loja'] == loja, :]

display(dicionario_lojas['Rio Mar Recife'].head())

,Código Venda,Data,ID Loja,Produto,Quantidade,Valor Unitário,Valor Final,Loja
62,46,2019-01-02,7,Camisa,1,100,100,Rio Mar Recife
63,46,2019-01-02,7,Calça Liso,2,190,380,Rio Mar Recife
64,46,2019-01-02,7,Cinto,1,200,200,Rio Mar Recife
113,87,2019-01-02,7,Camisa Estampa,1,113,113,Rio Mar Recife
142,109,2019-01-02,7,Camisa Gola V Listrado,3,116,348,Rio Mar Recife


In [5]:
# Definindo dia do indicador

dia_indicador = vendas['Data'].max()
print(dia_indicador)

2019-12-26 00:00:00


In [6]:
#Salvar as planilhas na pasta de backup
import pathlib

caminho_backup = pathlib.Path(r'Backup Arquivos Lojas')

arquivos_pasta_backup = caminho_backup.iterdir()

lista_nomes_backup = []
for arquivo in arquivos_pasta_backup:
    lista_nomes_backup.append(arquivo.name)

for loja in dicionario_lojas:
    if loja not in lista_nomes_backup:
        nova_pasta = caminho_backup / loja
        nova_pasta.mkdir()
    nome_arquivo = '{}_{}_{}.xlsx'.format(dia_indicador.month, dia_indicador.day, loja)
    local_arquivo = caminho_backup / loja / nome_arquivo
    dicionario_lojas[loja].to_excel(local_arquivo)



In [7]:
meta_faturamento_dia = 1000
meta_faturamento_ano = 1650000
meta_qtdeprodutos_dia = 4
meta_qtdeprodutos_ano = 120
meta_ticketmedio_dia = 500
meta_ticketmedio_ano = meta_ticketmedio_dia

In [8]:

import win32com.client as win32
import pythoncom
# Inicializa o COM na thread atual
pythoncom.CoInitialize()
outlook = win32.Dispatch('outlook.application')

for loja in dicionario_lojas:
    vendas_loja = dicionario_lojas[loja]
    vendas_loja_dia = vendas_loja.loc[vendas_loja['Data'] == dia_indicador, :]

    #Faturamento
    faturamento_ano = vendas_loja['Valor Final'].sum()
    faturamento_dia = vendas_loja_dia['Valor Final'].sum()

    #Diversidade de Produto
    diversidade_ano = len(vendas_loja['Produto'].unique())
    diversidade_dia = len(vendas_loja_dia['Produto'].unique())

    #Ticket Médio
    vendas_resumida = vendas_loja[['Código Venda', 'Valor Final']]
    vendas_resumida_ano = vendas_resumida.groupby('Código Venda').sum()
    vendas_resumida_dia = vendas_loja_dia[['Código Venda', 'Valor Final']].groupby('Código Venda').sum()


    ticket_medio_ano = vendas_resumida_ano['Valor Final'].mean()
    ticket_medio_dia = vendas_resumida_dia['Valor Final'].mean()


    #Email
    nome = emails.loc[emails['Loja'] == loja, 'Gerente'].values[0]
    mail = outlook.CreateItem(0)
    mail.To = emails.loc[emails['Loja'] == loja, 'E-mail'].values[0]
    mail.CC = 'silvapedrohenrique852@gmail.com'
    mail.Subject = f'OnePage Dia {dia_indicador.day}/{dia_indicador.month}/{dia_indicador.year} - Loja: {loja}'

    # ── Cenário (verde/vermelho) ────────────────────────────────────────────────
    cor_fat_dia    = 'green' if faturamento_dia   >= meta_faturamento_dia   else 'red'
    cor_fat_ano    = 'green' if faturamento_ano   >= meta_faturamento_ano   else 'red'
    cor_qtde_dia   = 'green' if diversidade_dia   >= meta_qtdeprodutos_dia  else 'red'
    cor_qtde_ano   = 'green' if diversidade_ano   >= meta_qtdeprodutos_ano  else 'red'
    cor_ticket_dia = 'green' if ticket_medio_dia  >= meta_ticketmedio_dia   else 'red'
    cor_ticket_ano = 'green' if ticket_medio_ano  >= meta_ticketmedio_ano   else 'red'

    # ── Outlook ─────────────────────────────────────────────────────────────────

    mail.HTMLBody = f'''
    <p>Bom dia, {nome}</p>
    <p>O resultado de ontem <strong>({dia_indicador.day}/{dia_indicador.month})</strong> da <strong>Loja {loja}</strong> foi:</p>

    <table>
    <tr><th>Indicador</th><th>Valor Dia</th><th>Meta Dia</th><th>Cenário Dia</th></tr>
    <tr>
        <td>Faturamento</td>
        <td style="text-align:center">R${faturamento_dia:.2f}</td>
        <td style="text-align:center">R${meta_faturamento_dia:.2f}</td>
        <td style="text-align:center"><font color="{cor_fat_dia}">◙</font></td>
    </tr>
    <tr>
        <td>Diversidade de Produtos</td>
        <td style="text-align:center">{diversidade_dia}</td>
        <td style="text-align:center">{meta_qtdeprodutos_dia}</td>
        <td style="text-align:center"><font color="{cor_qtde_dia}">◙</font></td>
    </tr>
    <tr>
        <td>Ticket Médio</td>
        <td style="text-align:center">R${ticket_medio_dia:.2f}</td>
        <td style="text-align:center">R${meta_ticketmedio_dia:.2f}</td>
        <td style="text-align:center"><font color="{cor_ticket_dia}">◙</font></td>
    </tr>
    </table>
    <br>
    <table>
    <tr><th>Indicador</th><th>Valor Ano</th><th>Meta Ano</th><th>Cenário Ano</th></tr>
    <tr>
        <td>Faturamento</td>
        <td style="text-align:center">R${faturamento_ano:.2f}</td>
        <td style="text-align:center">R${meta_faturamento_ano:.2f}</td>
        <td style="text-align:center"><font color="{cor_fat_ano}">◙</font></td>
    </tr>
    <tr>
        <td>Diversidade de Produtos</td>
        <td style="text-align:center">{diversidade_ano}</td>
        <td style="text-align:center">{meta_qtdeprodutos_ano}</td>
        <td style="text-align:center"><font color="{cor_qtde_ano}">◙</font></td>
    </tr>
    <tr>
        <td>Ticket Médio</td>
        <td style="text-align:center">R${ticket_medio_ano:.2f}</td>
        <td style="text-align:center">R${meta_ticketmedio_ano:.2f}</td>
        <td style="text-align:center"><font color="{cor_ticket_ano}">◙</font></td>
    </tr>
    </table>

    <p>Segue em anexo a planilha com todos os dados para mais detalhes.</p>
    <p>Qualquer dúvida estou à disposição.</p>
    <p>Att., Lira</p>
    '''

    attachment = pathlib.Path.cwd() / caminho_backup / loja / f'{dia_indicador.month}_{dia_indicador.day}_{loja}.xlsx'
    mail.Attachments.Add(str(attachment))

    mail.Send()
    

    